In [ ]:
import arcpy
import os
import random

# Set up workspace
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

# Processing step.
sinkhole_points = r"D:\path\to\sinkhole-risk-china\data\labels_Geological_hazard_in_China\selected_sinkhole_merge\Selected_Sinkhole_Merge_China.shp"
china_boundary = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"

# Step 1: Obtain China boundary range and spatial reference
print("...")
sr = arcpy.Describe(china_boundary).spatialReference
china_extent = arcpy.Describe(china_boundary).extent

# Set the output coordinate system
arcpy.env.outputCoordinateSystem = sr

# Step 2: Create a polygonal fishing net (unit size 10km)
print("Creating fishing net...")
fishnet = os.path.join(scratch_gdb, "fishnet")
cell_size = 10000  # 10km

# Create a polygonal fishing net
# Check whether the field names are consistent.
arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=f"{china_extent.XMin} {china_extent.YMin}",
    y_axis_coord=f"{china_extent.XMin} {china_extent.YMax}",
    cell_width=cell_size,
    cell_height=cell_size,
    corner_coord=f"{china_extent.XMax} {china_extent.YMax}",
    geometry_type="POLYGON"
)

# Step 3: Generate the center point of the fishing net
print("Generating candidate points...")
candidate_points = os.path.join(scratch_gdb, "candidate_points")
arcpy.management.FeatureToPoint(fishnet, candidate_points, "CENTROID")  # Use centroids instead of interior points

# Step 4: Crop to China border
print("...")
clipped_points = os.path.join(scratch_gdb, "clipped_points")
arcpy.analysis.Clip(candidate_points, china_boundary, clipped_points)

# Step 5: Create a 10km buffer of positive samples
print("Create buffer...")
sinkhole_buffer = os.path.join(scratch_gdb, "sinkhole_buffer")
arcpy.analysis.Buffer(sinkhole_points, sinkhole_buffer, "10 Kilometers")

# Step 6: Screen effective candidate points
print("Filter candidate points...")
valid_points = os.path.join(scratch_gdb, "valid_points")
arcpy.analysis.Erase(clipped_points, sinkhole_buffer, valid_points)

# Step 7: Randomly select the same number of points as the positive samples
print("Randomly select negative samples...")
num_samples = int(arcpy.management.GetCount(sinkhole_points)[0])
all_points = [row[0] for row in arcpy.da.SearchCursor(valid_points, ["OID@"])]

# Make sure there are enough candidate points
if len(all_points) < num_samples:
    print(f"Warning: Number of valid candidate points ({len(all_points)})({num_samples})")
    print("It may be necessary to increase the net unit size or reduce the buffer distance")
    selected_ids = all_points
else:
    selected_ids = random.sample(all_points, num_samples)

# Processing step.
negative_samples = os.path.join(scratch_gdb, "negative_samples")
arcpy.management.MakeFeatureLayer(valid_points, "temp_layer")
arcpy.management.SelectLayerByAttribute("temp_layer", "NEW_SELECTION", f"OBJECTID IN ({','.join(map(str, selected_ids))})")
arcpy.management.CopyFeatures("temp_layer", negative_samples)

# 8:
print("Add results to map...")
project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
active_map.addDataFromPath(negative_samples)

print(f"Successfully generated{len(selected_ids)}negative sample points!")

In [ ]:
# New step: Add and calculate latitude and longitude fields (using projection and cursor)
print("Add latitude and longitude fields...")
wgs84 = arcpy.SpatialReference(4326)  # WGS84 coordinate system

# Project negative sample points to WGS84 coordinate system
projected_negative_samples = os.path.join(scratch_gdb, "projected_negative_samples")
arcpy.management.Project(negative_samples, projected_negative_samples, wgs84)

# Processing step.
arcpy.management.AddField(projected_negative_samples, "Longitude", "DOUBLE")
arcpy.management.AddField(projected_negative_samples, "Latitude", "DOUBLE")

# Use cursor to calculate latitude and longitude
with arcpy.da.UpdateCursor(projected_negative_samples, ["SHAPE@X", "SHAPE@Y", "Longitude", "Latitude"]) as cursor:
    for row in cursor:
        row[2] = row[0]  # X -> Longitude
        row[3] = row[1]  # Y -> Latitude
        cursor.updateRow(row)

# Update negative samples to projected points
negative_samples = projected_negative_samples

# 8:
print("Add results to map...")
project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
active_map.addDataFromPath(negative_samples)

print(f"Successfully generated{len(selected_ids)}negative sample points, and added latitude and longitude information!")